VENIMOS DEL ARCHIVO .C:/Users/Josue/4GA.Datascience/src/PASO5/VariablesTarget/testing/HappyLrscalePolintr/NewDataModel.ipynb.ipynb donde hemos recogido mas variables dado que han sido insuficientes para obtener un score decente por el cual empezar, scores inferiores a 0.5 accuracy tanto de modelos de clasificacion como de regresion.


--------------------------------------------------------------------------


En este archivo hemos transformado las variables entrantes , casi todas en escalas ordinarias de rangos 0-10, 0-8,0-6, a escalas de categorias mas reducidas en algunos casos, ademas, hemos creado contadores
en funcion de las respuestas invalidas de las muestras, contadores neutros, contando la neutralidad en los valores de las variables, y contadores positivos para las muestras propensas a posicionarse en extremos.
Hemos arreglado datos, normalizando, pasando categorias a valores discretos factorizados, hemos limpiado datos, hemos utilizado datos de post stratificacion dados por la web y pesos ponderados de unicos para cada muestra por estrato social y pais, etc.
COn un total de 80 variables base, muchas de ellas duplicadas por la fecha de registro, hemos conseguido bastantes variables extras por combinacion logica entre ellas entre otras cosas.
Vamos a realizar un modelo de clasificacion para las Variables 'lrscale' o una de sus variantes y para la variable 'Happy' o una de sus variantes.

Primero de todo vamos a utilizar un modelo base de XGBooost con el grid de hyperopt para tantear alrededor de unos hiperparametros base.

PARA LA VARIABLE HAPPY:

In [66]:
import numpy as np, random
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.model_selection import KFold, cross_val_score
from sklearn.utils import resample
random.seed(42)

def xgboost_numericas(X_train, y_train, X_test, y_test, ruta_guardado=None, max_evals=20, max_features=None):


    space = {
    'max_depth': hp.quniform('max_depth', 15, 20, 1),
    'learning_rate': hp.uniform('learning_rate', 0.10, 0.13),
    'n_estimators': hp.quniform('n_estimators', 100, 200, 10),
    'subsample': hp.uniform('subsample', 0.85, 0.90),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.42, 0.50),
    'min_child_weight': hp.quniform('min_child_weight', 6, 10, 1),
    'gamma': hp.uniform('gamma', 0.55, 0.65),
    'reg_alpha': hp.loguniform('reg_alpha', np.log(0.2), np.log(0.3)),
    'reg_lambda': hp.loguniform('reg_lambda', np.log(0.05), np.log(0.06)),
    'scale_pos_weight': hp.uniform('scale_pos_weight', 1, 10)
}

    def objective(params):
        modelo = xgb.XGBClassifier(
            max_depth=int(params['max_depth']),
            learning_rate=params['learning_rate'],
            n_estimators=int(params['n_estimators']),
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            min_child_weight=int(params['min_child_weight']),
            gamma=params['gamma'],
            random_state=42,
            eval_metric='logloss'
        )
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        score = accuracy_score(y_test, y_pred)
        return {'loss': -score, 'status': STATUS_OK}

    trials = Trials()
    best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=max_evals, trials=trials)

    mejor_modelo = xgb.XGBClassifier(
        max_depth=int(best['max_depth']),
        learning_rate=best['learning_rate'],
        n_estimators=int(best['n_estimators']),
        subsample=best['subsample'],
        colsample_bytree=best['colsample_bytree'],
        min_child_weight=int(best['min_child_weight']),
        gamma=best['gamma'],
        random_state=42,
        eval_metric='logloss',
        enable_categorical=True,
    )
    mejor_modelo.fit(X_train, y_train, verbose=False)

    
    if max_features is not None and max_features < X_train.shape[1]:
        feature_importance = mejor_modelo.feature_importances_
        indices_ordenados = np.argsort(feature_importance)[::-1]
        selected_features = X_train.columns[indices_ordenados[:max_features]]
        X_train = X_train[selected_features]
        X_test = X_test[selected_features]
        mejor_modelo.fit(X_train, y_train, verbose=False)

    y_pred = mejor_modelo.predict(X_test)
    y_pred_proba = mejor_modelo.predict_proba(X_test)
    auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
    accuracy = accuracy_score(y_test, y_pred)

    
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores = cross_val_score(mejor_modelo, X_train, y_train, cv=kf, scoring='accuracy')

    print("\nMejor modelo encontrado (XGBoost Hyperopt con Selección de Características):")
    print(f"best_params: {best}")
    print(f"AUC: {auc}")
    print(f"Accuracy: {accuracy}")
    print(f"Cross-Validation Accuracy: {np.mean(cv_scores)}")

    if ruta_guardado:
        joblib.dump(mejor_modelo, f"{ruta_guardado}mejor_modelo_xgboost_numericas.pkl")
        with open(f"{ruta_guardado}metricas_xgboost_numericas.txt", "w") as f:
            f.write(f"best_params: {best}\n")
            f.write(f"AUC: {auc}\n")
            f.write(f"Accuracy: {accuracy}\n")
            f.write(f"Cross-Validation Accuracy: {np.mean(cv_scores)}\n")
            f.write(f"Predictoras utilizadas: {list(X_train.columns) if isinstance(X_train, pd.DataFrame) else list(range(X_train.shape[1]))}\n")
            f.write(f"Variable objetivo: {y_train.name if hasattr(y_train, 'name') else 'desconocido'}\n")

    return mejor_modelo

X_train, X_test, y1_train, y1_test = joblib.load("C:/Users/Josue/4GA.Datascience/src/PASO5/VariablesTarget/testing/HappyLrscalePolintr/SplitHappy3c.pkl")
X_train_sampled, y1_train_sampled = resample(X_train, y1_train, n_samples=20000, random_state=42)
Happy3cModel = xgboost_numericas(X_train_sampled, y1_train_sampled, X_test, y1_test, ruta_guardado="C:/Users/Josue/4GA.DataScience/models/", max_features=12)

100%|██████████| 20/20 [00:50<00:00,  2.51s/trial, best loss: -0.71616]           

Mejor modelo encontrado (XGBoost Hyperopt con Selección de Características):
best_params: {'colsample_bytree': np.float64(0.47753743778504554), 'gamma': np.float64(0.6482038581352298), 'learning_rate': np.float64(0.12326717718013894), 'max_depth': np.float64(19.0), 'min_child_weight': np.float64(6.0), 'n_estimators': np.float64(180.0), 'reg_alpha': np.float64(0.24872126993839974), 'reg_lambda': np.float64(0.05491930763496308), 'scale_pos_weight': np.float64(5.485975822889809), 'subsample': np.float64(0.88022768504974)}
AUC: 0.8524697231377849
Accuracy: 0.7019733333333333
Cross-Validation Accuracy: 0.73475


Vamos a utilizar RandomForest con un grid de Bayesian.

In [61]:
import numpy as np, random
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib
from sklearn.model_selection import KFold, cross_val_score
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
random.seed(42)

def random_forest_bayes_categoricas(X_train, y_train, X_test, y_test, ruta_guardado=None, n_iter=20, max_features=None):
    """
    Entrena un modelo Random Forest con BayesSearchCV y selección automática de características,
    específicamente para variables categóricas.
    """

    space = {
    'n_estimators': Integer(200, 1000),
    'max_depth': Integer(8, 25),
    'min_samples_split': Integer(2, 50),
    'min_samples_leaf': Integer(1, 30),
    'max_features': Categorical(['sqrt', 'log2']),
    'criterion': Categorical(['gini', 'entropy']),
    'max_samples': Real(0.8, 1.0, prior="uniform"),
    'bootstrap': Categorical([True, False])
        }

    modelo = RandomForestClassifier(random_state=42)

    bayes_search = BayesSearchCV(
        modelo,
        space,
        n_iter=n_iter,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='accuracy',
        random_state=42
    )

    bayes_search.fit(X_train, y_train)

    mejor_modelo = bayes_search.best_estimator_

  
    if max_features is not None and max_features < X_train.shape[1]:
        feature_importance = mejor_modelo.feature_importances_
        indices_ordenados = np.argsort(feature_importance)[::-1]
        selected_features = X_train.columns[indices_ordenados[:max_features]]
        X_train = X_train[selected_features]
        X_test = X_test[selected_features]
        mejor_modelo.fit(X_train, y_train)

    y_pred = mejor_modelo.predict(X_test)
    y_pred_proba = mejor_modelo.predict_proba(X_test)
    auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
    accuracy = accuracy_score(y_test, y_pred)

   
    kf = KFold(n_splits=6, shuffle=True, random_state=42)
    cv_scores = cross_val_score(mejor_modelo, X_train, y_train, cv=kf, scoring='accuracy')

    print("\nModelo Random Forest con BayesSearchCV y Selección de Características:")
    print(f"best_params: {bayes_search.best_params_}")
    print(f"AUC: {auc}")
    print(f"Accuracy: {accuracy}")
    print(f"Cross-Validation Accuracy: {np.mean(cv_scores)}")

    if ruta_guardado:
        joblib.dump(mejor_modelo, f"{ruta_guardado}mejor_modelo_rf_bayes_categoricas.pkl")
        with open(f"{ruta_guardado}metricas_rf_bayes_categoricas.txt", "w") as f:
            f.write(f"best_params: {bayes_search.best_params_}\n")
            f.write(f"AUC: {auc}\n")
            f.write(f"Accuracy: {accuracy}\n")
            f.write(f"Cross-Validation Accuracy: {np.mean(cv_scores)}\n")
            f.write(f"Predictoras utilizadas: {list(X_train.columns) if isinstance(X_train, pd.DataFrame) else list(range(X_train.shape[1]))}\n")
            f.write(f"Variable objetivo: {y_train.name if hasattr(y_train, 'name') else 'desconocido'}\n")

    return mejor_modelo

# Ejemplo de uso:
X_train, X_test, y1_train, y1_test = joblib.load("C:/Users/Josue/4GA.Datascience/src/PASO5/VariablesTarget/testing/HappyLrscalePolintr/SplitHappy3c.pkl")
X_train_sampled, y1_train_sampled = resample(X_train, y1_train, n_samples=6000, random_state=42)
Happy3cModelRF = random_forest_bayes_categoricas(X_train_sampled, y1_train_sampled, X_test, y1_test, ruta_guardado="C:/Users/Josue/4GA.DataScience/models/", n_iter=30, max_features=10)


Modelo Random Forest con BayesSearchCV y Selección de Características:
best_params: OrderedDict({'bootstrap': True, 'criterion': 'entropy', 'max_depth': 12, 'max_features': 'sqrt', 'max_samples': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 258})
AUC: 0.8463609682185677
Accuracy: 0.7031466666666667
Cross-Validation Accuracy: 0.7326666666666667


PARA LA VARIABLE LRSCALE USAREMOS SVC, PREVIAMENTE TANTEADOS ES EL QUE MEJOR SCORE EN DEFAULT ME HA DADO.

In [83]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib
from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold
from sklearn.utils import resample
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real

def xgboost_logistic_bayes_categoricas(X_train, y_train, X_test, y_test, ruta_guardado=None, n_iter=20, max_features=None):
    """
    Entrena un modelo XGBoost para regresión logística con BayesSearchCV y selección automática de características.
    """

    space = {
        'n_estimators': Integer(100, 1000),
        'max_depth': Integer(3, 10),
        'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
        'subsample': Real(0.8, 1.0, prior='uniform'),
        'colsample_bytree': Real(0.8, 1.0, prior='uniform'),
        'gamma': Real(0, 1, prior='uniform'),
        'min_child_weight': Integer(1, 10),
        'reg_alpha': Real(1e-5, 1, prior='log-uniform'),
        'reg_lambda': Real(1e-5, 1, prior='log-uniform'),
    }

    modelo = xgb.XGBClassifier(
        objective='binary:logistic',  # Para regresión logística binaria
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )

    bayes_search = BayesSearchCV(
        modelo,
        space,
        n_iter=n_iter,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc',  # Usamos ROC AUC para regresión logística
        random_state=42
    )

    bayes_search.fit(X_train, y_train)

    mejor_modelo = bayes_search.best_estimator_

    if max_features is not None and max_features < X_train.shape[1]:
        feature_importance = mejor_modelo.feature_importances_
        indices_ordenados = np.argsort(feature_importance)[::-1]
        selected_features = X_train.columns[indices_ordenados[:max_features]]
        X_train = X_train[selected_features]
        X_test = X_test[selected_features]
        mejor_modelo.fit(X_train, y_train)

    y_pred_proba = mejor_modelo.predict_proba(X_test)[:, 1]  # Probabilidades para la clase positiva
    auc = roc_auc_score(y_test, y_pred_proba)
    y_pred = (y_pred_proba > 0.5).astype(int)  # Convertir probabilidades a predicciones binarias
    accuracy = accuracy_score(y_test, y_pred)

    kf = KFold(n_splits=6, shuffle=True, random_state=42)
    cv_scores = cross_val_score(mejor_modelo, X_train, y_train, cv=kf, scoring='roc_auc')

    print("\nModelo XGBoost para Regresión Logística con BayesSearchCV y Selección de Características:")
    print(f"best_params: {bayes_search.best_params_}")
    print(f"AUC: {auc}")
    print(f"Accuracy: {accuracy}")
    print(f"Cross-Validation AUC: {np.mean(cv_scores)}")

    if ruta_guardado:
        joblib.dump(mejor_modelo, f"{ruta_guardado}mejor_modelo_xgb_logistic_bayes_categoricas.pkl")
        with open(f"{ruta_guardado}metricas_xgb_logistic_bayes_categoricas.txt", "w") as f:
            f.write(f"best_params: {bayes_search.best_params_}\n")
            f.write(f"AUC: {auc}\n")
            f.write(f"Accuracy: {accuracy}\n")
            f.write(f"Cross-Validation AUC: {np.mean(cv_scores)}\n")
            f.write(f"Predictoras utilizadas: {list(X_train.columns) if isinstance(X_train, pd.DataFrame) else list(range(X_train.shape[1]))}\n")
            f.write(f"Variable objetivo: {y_train.name if hasattr(y_train, 'name') else 'desconocido'}\n")

    return mejor_modelo

# Ejemplo de uso:
X_train, X_test, y1_train, y1_test = joblib.load("C:/Users/Josue/4GA.Datascience/src/PASO5/VariablesTarget/testing/HappyLrscalePolintr/SplitLrscale5c.pkl")
X_train_sampled, y1_train_sampled = resample(X_train, y1_train, n_samples=6000, random_state=42)

Happy3cXGBLog = xgboost_logistic_bayes_categoricas(X_train_sampled, y1_train_sampled, X_test, y1_test, ruta_guardado="C:/Users/Josue/4GA.DataScience/models/", n_iter=30, max_features=10)

c:\Users\Josue\4GA.Datascience\.venvSVM\Lib\site-packages\xgboost\training.py:183: UserWarning: [05:17:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


ValueError: multi_class must be in ('ovo', 'ovr')

In [1]:
import pandas as pd
import joblib
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold, cross_val_score
from skopt import gp_minimize
from skopt.space import Integer, Real
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif
import os

def entrenar_y_evaluar_svc_multiclase(X_train, y_train, X_test, y_test, max_features=None, ruta_guardado=None):
    """
    Entrena y evalúa un modelo SVC para clasificación multiclase con selección de características y guarda el modelo si se proporciona la ruta.
    """

    original_columns = X_train.columns

    if max_features and max_features < X_train.shape[1]:
        selector = SelectKBest(f_classif, k=max_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)
        selected_features = original_columns[selector.get_support(indices=True)]
    else:
        selected_features = original_columns

    space = {
        'C': Real(1e-3, 1e3, prior='log-uniform'),
        'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'degree': Integer(2, 5),
        'gamma_type': ['scale', 'auto', 'number'],
        'gamma_number': Real(1e-3, 1e3, prior='log-uniform'),
        'coef0': Real(-1, 1),
    }

    def gamma_selector(gamma_type, gamma_number):
        if gamma_type == 'number':
            return gamma_number
        else:
            return gamma_type

    def objective(params):
        C, kernel, degree, gamma_type, gamma_number, coef0 = params

        gamma = gamma_selector(gamma_type, gamma_number)

        model = SVC(C=C, kernel=kernel, degree=degree, gamma=gamma, coef0=coef0, probability=True, random_state=42)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc_ovr') # Cambiado a roc_auc_ovr
        return -scores.mean()

    result = gp_minimize(objective, list(space.values()), n_calls=50, random_state=42, verbose=True)

    best_params = {key: value for key, value in zip(space.keys(), result.x)}

    final_model = SVC(C=best_params['C'], kernel=best_params['kernel'], degree=best_params['degree'], 
                      gamma=gamma_selector(best_params['gamma_type'], best_params['gamma_number']), 
                      coef0=best_params['coef0'], probability=True, random_state=42)

    final_model.fit(X_train, y_train)

    y_pred_proba = final_model.predict_proba(X_test)
    y_pred = final_model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr') # Cambiado a multi_class='ovr'
    accuracy = accuracy_score(y_test, y_pred)

    print("\nResultados Finales:")
    print(f"Mejores parámetros: {best_params}")
    print(f"AUC en el conjunto de prueba: {auc}")
    print(f"Accuracy en el conjunto de prueba: {accuracy}")
    print(f"Variables utilizadas: {list(selected_features)}")

    if ruta_guardado:
        os.makedirs(os.path.dirname(ruta_guardado), exist_ok=True)
        joblib.dump(final_model, ruta_guardado)
        print(f"Modelo guardado en: {ruta_guardado}")

# Ejemplo de uso:
X_train, X_test, y1_train, y1_test = joblib.load("C:/Users/Josue/4GA.Datascience/src/PASO5/VariablesTarget/testing/HappyLrscalePolintr/SplitLrscale5c.pkl")
X_train_sampled, y1_train_sampled = resample(X_train, y1_train, n_samples=3000, random_state=42)

entrenar_y_evaluar_svc_multiclase(X_train_sampled, y1_train_sampled, X_test, y1_test, max_features=6, ruta_guardado="C:/Users/Josue/4GA.DataScience/models/svc_model_multiclase.pkl")

Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 123.8314
Function value obtained: -0.5925
Current minimum: -0.5925
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 4.3296
Function value obtained: -0.5976
Current minimum: -0.5976
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 412.7216
Function value obtained: -0.5925
Current minimum: -0.5976
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 5.1539
Function value obtained: -0.5717
Current minimum: -0.5976
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 5.1601
Function value obtained: -0.5736
Current minimum: -0.5976
Iteration No: 6 start

: 

: 